# Student Performance Prediction using Backpropagation Neural Network (BPNN)

**Course / Project:** Machine Learning — Binary Classification

This notebook builds a **feed-forward neural network** trained with **backpropagation** (implemented via TensorFlow/Keras) to predict whether a student **Passes** or **Fails** (`result`: 0 = Fail, 1 = Pass).

**Highlights**
- Configurable **feature column names** (academic, mental well-being, and social/family support proxies).
- **Reproducible** workflows (random seeds).
- **Train / validation / test** evaluation with **ROC–AUC**, confusion matrix, and calibration-style probability outputs.
- **Leakage-aware scaling:** `MinMaxScaler` is **fit on training data only**, then applied to validation and test splits.



### 1. 📥 Import Libraries

We use **NumPy / Pandas** for data handling, **Matplotlib / Seaborn** for presentation-quality plots, **scikit-learn** for splitting, preprocessing, and metrics, and **TensorFlow / Keras** for the BPNN.



In [ ]:
# --- Core numerics & tables ---
import os
import random
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- sklearn: split, preprocessing, metrics ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    classification_report,
)

# --- TensorFlow / Keras (backpropagation is handled by the optimizer) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# --- Persistence ---
import joblib

# --- Notebook display ---
from IPython.display import display

# Presentation-friendly defaults (style name varies by matplotlib version)
for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue
sns.set_context("talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)



### 2. 📂 Load Dataset

We load `student_data.csv`. The column names below are centralized so you can swap datasets/features without rewriting the entire notebook.

**Default feature groups (influential academically + mentally + socially)**
- **Academic workload & outcomes:** `study_hours`, `attendance`, `previous_marks`, `assignments`
- **Lifestyle / mental-health proxy:** `sleep_hours`, `mental_stress`
- **Social / family support proxy:** `parent_involvement` (parental engagement / involvement score)

**Target**
- `result` where **0 = Fail** and **1 = Pass**



In [ ]:
# =========================
# CONFIG (edit in one place)
# =========================
RANDOM_SEED = 42

# Reproducibility (as far as TF allows across hardware)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

DATA_PATH = "student_data.csv"  # change if your file name/path differs

# Feature columns used for modeling (easy to modify)
FEATURE_COLUMNS = [
    # Academic
    "study_hours",
    "attendance",
    "previous_marks",
    "assignments",
    # Lifestyle / mental proxy
    "sleep_hours",
    "mental_stress",
    # Social / family support proxy
    "parent_involvement",
]

TARGET_COLUMN = "result"  # 0 = Fail, 1 = Pass

# Load
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nInfo:")
print(df.info())
print("\nMissing values (per column):")
print(df.isna().sum().sort_values(ascending=False).head(20))



### 3. 🧹 Data Preprocessing

**Goals**
- Handle missing values (drop rows for simplicity; alternatively impute means).
- Encode categorical variables if present (none in the default CSV, but the pipeline is included for portability).
- Prepare a clean modeling matrix `X` and label vector `y`.

**Note on scaling:** We intentionally **delay** `MinMaxScaler` until after the train/validation/test split to avoid information leakage from validation/test into the scaler statistics.



In [ ]:
# Work on a copy
data = df.copy()

# Quick sanity: required columns exist
missing_cols = [c for c in (FEATURE_COLUMNS + [TARGET_COLUMN]) if c not in data.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# Missing values: drop rows with any missing feature/target (simple + common for class demos)
before = len(data)
data = data.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).reset_index(drop=True)
after = len(data)
print(f"Dropped {before - after} rows with missing values (if any). Remaining: {after}")

# If you add categorical columns later, list them here:
CATEGORICAL_COLUMNS = []  # e.g., ["gender", "school_type"]

label_encoders = {}
for col in CATEGORICAL_COLUMNS:
    if col not in data.columns:
        continue
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

X = data[FEATURE_COLUMNS].copy()
y = data[TARGET_COLUMN].astype(int).values

# Basic label integrity check
assert set(np.unique(y)).issubset({0, 1}), "Target must be binary {0,1}."

print("X shape:", X.shape)
print("Class balance (0/1):", pd.Series(y).value_counts().to_dict())



### 4. 🧠 Feature Engineering (Advanced)

We construct an interpretable composite score:

\[
\texttt{performance\_index} =
w_1\cdot\widehat{\texttt{previous\_marks}} +
w_2\cdot\widehat{\texttt{attendance}} +
w_3\cdot\widehat{\texttt{assignments}}
\]

where \(\widehat{\cdot}\) denotes **min–max normalization to [0,1] per column** *computed on the training split only* (implemented in the next step after splitting).

**Additional engineering used here (before splitting)**
- **Stress–sleep interaction:** `stress_sleep = mental_stress * sleep_hours` (captures fatigue under stress; scale-sensitive, so it will be min–max scaled later).
- **Engagement ratio:** `engagement = attendance * study_hours` (proxy for consistent participation + effort).

These features are widely used as *simple, interpretable* proxies in student success modeling; you can refine them with domain knowledge.



In [ ]:
# Feature engineering on the full pre-split frame (no scaler leakage for the main features yet)
X_fe = X.copy()

# Safe raw-derived features (will be scaled after split)
X_fe["engagement"] = X_fe["attendance"] * X_fe["study_hours"]
X_fe["stress_sleep"] = X_fe["mental_stress"] * X_fe["sleep_hours"]

# performance_index will be created AFTER train split using train-only min-max stats
# We'll store the recipe as a function:

def add_performance_index(X_df: pd.DataFrame, mins: pd.Series, maxs: pd.Series, weights=None) -> pd.DataFrame:
    """Weighted index from min-max normalized columns (mins/maxs come from TRAIN only)."""
    if weights is None:
        weights = {"previous_marks": 0.5, "attendance": 0.3, "assignments": 0.2}

    out = X_df.copy()
    eps = 1e-8
    for c in weights.keys():
        if c not in out.columns:
            raise KeyError(f"Missing column for performance_index: {c}")

    norm = {}
    for c, w in weights.items():
        norm[c] = (out[c] - float(mins[c])) / (float(maxs[c] - mins[c]) + eps)

    out["performance_index"] = sum(weights[c] * norm[c] for c in weights.keys())
    return out


print("Engineered columns (pre-index):", list(X_fe.columns))



### 5. 🔀 Train–Validation–Test Split (60% / 20% / 20%)

We split twice:
1. Hold out **20%** as the final **test** set.
2. From the remaining **80%**, hold out **25%** as **validation** (because \(0.25 \times 0.8 = 0.2\) of all samples).

This yields **60% train / 20% validation / 20% test**.



In [ ]:
# First split: train+val vs test (80/20)
X_tv, X_test, y_tv, y_test = train_test_split(
    X_fe,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

# Second split: train vs val (75/25 of the 80% block => 60/20 overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv,
    y_tv,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_tv,
)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

# --- Train-only stats for performance_index ---
idx_cols = ["previous_marks", "attendance", "assignments"]
mins = X_train[idx_cols].min()
maxs = X_train[idx_cols].max()

X_train = add_performance_index(X_train, mins=mins, maxs=maxs)
X_val = add_performance_index(X_val, mins=mins, maxs=maxs)
X_test = add_performance_index(X_test, mins=mins, maxs=maxs)

# --- MinMax scaling (fit on TRAIN only) ---
FEATURES_FOR_MODEL = list(X_train.columns)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Model input dim:", X_train_s.shape[1])
print("Features:", FEATURES_FOR_MODEL)



### 6. 🧠 Build BPNN Model (Backpropagation Neural Network)

We use a **fully connected (Dense)** architecture with **BatchNormalization** for stable optimization, **Dropout** for regularization, and a **sigmoid** output for binary classification.

**Backpropagation** is performed automatically by Keras during `fit()` using automatic differentiation (`GradientTape` under the hood) with the Adam optimizer.



In [ ]:
INPUT_DIM = X_train_s.shape[1]

model = models.Sequential(
    [
        layers.Input(shape=(INPUT_DIM,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="student_pass_fail_bpnn",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()



### 7. 🚀 Train Model

We train for up to **100 epochs** with a **validation** monitor and **EarlyStopping** to prevent overfitting and save time.



In [ ]:
EPOCHS = 100
BATCH_SIZE = 256

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    )
]

history = model.fit(
    X_train_s,
    y_train,
    validation_data=(X_val_s, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)



### 8. 📊 Model Evaluation (Test Set)

We report standard classification metrics and **ROC–AUC** using predicted probabilities.



In [ ]:
y_proba = model.predict(X_test_s, verbose=0).reshape(-1)
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, digits=4))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)



### 9. 📈 Visualization

We visualize optimization curves, the confusion matrix, and the ROC curve.



In [ ]:
hist = history.history

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(hist["loss"], label="Train loss")
ax[0].plot(hist["val_loss"], label="Val loss")
ax[0].set_title("Training vs Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Binary cross-entropy")
ax[0].legend()

ax[1].plot(hist["accuracy"], label="Train accuracy")
ax[1].plot(hist["val_accuracy"], label="Val accuracy")
ax[1].set_title("Training vs Validation Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
    xticklabels=["Pred Fail", "Pred Pass"],
    yticklabels=["True Fail", "True Pass"],
    ax=ax,
)
ax.set_title("Confusion Matrix (Test)")
plt.tight_layout()
plt.show()

fpr, tpr, thr = roc_curve(y_test, y_proba)
plt.figure(figsize=(7.5, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC Curve (Test)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()



### 10. 💾 Save Model

We save:
- `model.h5`: trained Keras model
- `scaler.pkl`: fitted scaler + metadata needed to reproduce preprocessing for new students



In [ ]:
MODEL_PATH = "model.h5"
SCALER_PATH = "scaler.pkl"

model.save(MODEL_PATH)

artifact = {
    "scaler": scaler,
    "feature_columns": FEATURE_COLUMNS,
    "features_for_model": FEATURES_FOR_MODEL,
    "target_column": TARGET_COLUMN,
    "performance_index_weights": {"previous_marks": 0.5, "attendance": 0.3, "assignments": 0.2},
    "performance_index_mins": mins,
    "performance_index_maxs": maxs,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "label_encoders": label_encoders,
    "random_seed": RANDOM_SEED,
}

joblib.dump(artifact, SCALER_PATH)

print("Saved:", MODEL_PATH)
print("Saved:", SCALER_PATH)



### 11. 🔮 Prediction Function (New Student)

This helper applies the **same preprocessing recipe** (feature engineering + `performance_index` using train-derived bounds + `MinMaxScaler`) and returns:
- predicted class (**Pass/Fail**)
- **confidence** as the predicted probability of **Pass** (`P(pass)`)



In [ ]:
def predict_student_pass_fail(student: dict, model_path=MODEL_PATH, artifact_path=SCALER_PATH, threshold: float = 0.5):
    """Predict pass/fail for one student.

    Parameters
    ----------
    student : dict
        Keys should include the raw FEATURE_COLUMNS used during training.
    threshold : float
        If predicted probability of pass >= threshold => Pass.

    Returns
    -------
    dict with keys: prediction_label, pass_probability, fail_probability
    """
    art = joblib.load(artifact_path)
    mdl = keras.models.load_model(model_path)

    scaler_ = art["scaler"]
    feats = art["feature_columns"]
    feats_model = art["features_for_model"]
    mins_ = art["performance_index_mins"]
    maxs_ = art["performance_index_maxs"]
    wts = art["performance_index_weights"]

    # Build a single-row frame in the correct raw-feature order
    row = {k: float(student[k]) for k in feats}
    X0 = pd.DataFrame([row])

    # Same engineered features as training (before performance_index)
    X0["engagement"] = X0["attendance"] * X0["study_hours"]
    X0["stress_sleep"] = X0["mental_stress"] * X0["sleep_hours"]

    # performance_index with train-derived bounds
    eps = 1e-8
    norm = {}
    for c in wts.keys():
        norm[c] = (X0[c] - float(mins_[c])) / (float(maxs_[c] - mins_[c]) + eps)
    X0["performance_index"] = float(sum(wts[c] * float(norm[c].iloc[0]) for c in wts.keys()))

    # Align columns to training order
    X0 = X0.reindex(columns=list(feats_model), fill_value=0.0)
    Xs = scaler_.transform(X0)

    p_pass = float(mdl.predict(Xs, verbose=0).reshape(-1)[0])
    label = "Pass" if p_pass >= threshold else "Fail"

    return {
        "prediction_label": label,
        "pass_probability": p_pass,
        "fail_probability": 1.0 - p_pass,
    }


# Example (uses approximate averages; replace with real inputs)
example_student = {
    "study_hours": 3.0,
    "attendance": 0.85,
    "previous_marks": 72.0,
    "assignments": 70.0,
    "sleep_hours": 6.5,
    "mental_stress": 5.0,
    "parent_involvement": 6.0,
}

print(predict_student_pass_fail(example_student))



### References / Notes for Viva

- **BPNN:** multi-layer perceptron trained by gradient-based optimization; Keras performs **backpropagation** automatically.
- **Why stratified splitting?** keeps similar class proportions across splits for more stable metrics.
- **Why not scale before splitting?** scaler statistics should not include validation/test samples.
- **Interpreting probabilities:** sigmoid outputs are **model confidences**, not calibrated real-world probabilities unless you apply calibration methods.

---
**End of notebook**

